# Jacobian-guided spectral scheduling

We study a three-layer linear network
$$
\hat A = W_3 W_2 W_1,
\qquad
\hat A \approx A.
$$
Each training step computes the ordinary gradient for every layer, but before applying the update we may reshape the update spectrum. For a raw update matrix $U$, define
$$
\operatorname{spec}_p(U)
=
P\operatorname{diag}(\sigma_i^p)Q^\top
\cdot
\frac{\|U\|_F}{\|P\operatorname{diag}(\sigma_i^p)Q^\top\|_F},
\qquad
U=P\operatorname{diag}(\sigma_i)Q^\top.
$$
The exponent $p$ controls the trade-off:

- $p=1$: standard SGD update.
- $p=0.5$: partially flattens the update spectrum.
- $p=0$: uses an orthogonalized update direction with the same Frobenius norm.

The action at a step is a schedule $a=(p_1,p_2,p_3)\in\{1,0.5,0\}^3$, one exponent per layer. We compare three ways to choose this schedule:

- **sgd**: always uses $(1,1,1)$.
- **dp_rollout**: every $T$ steps, simulates each candidate schedule for $H$ steps and chooses the one with the lowest rollout terminal loss.
- **dp_jacobian**: every $T$ steps, scores each candidate by combining its one-step loss with a Jacobian-based prediction of future convergence speed.

The goal is not only to find a better schedule, but to predict when a spectrum-shaping trade-off is worth paying for: a schedule may hurt the immediate loss while improving the local geometry enough to make later optimization faster.

This viewpoint is closely related to the difference between a greedy algorithm and dynamic programming. A greedy rule chooses the action that gives the largest immediate decrease. In this setting, that means choosing the schedule with the smallest one-step loss. Dynamic programming instead asks which action moves the system into a state with the best future value. Our scheduling problem has exactly this form: the action is not only an update to reduce the current loss, but also a transition to a new set of weights whose geometry may make future optimization easier or harder.

The Jacobian value estimate below is a local, differentiable approximation to that future value. It does not merely ask whether the current loss drops fastest; it asks whether the new state has a larger effective contraction rate for subsequent training. In other words, spectral scheduling is a search for a better transition, not just a search for the steepest immediate descent.



## Experiment setup

All three methods start from the same target matrix and the same initial weights. The only difference is how the spectral schedule is selected. The rollout method is an empirical planning baseline; the Jacobian method replaces most of that empirical search with a local value estimate.


In [ ]:
import math
from collections import Counter
from itertools import product as cartesian_product

import matplotlib.pyplot as plt
import torch

torch.set_default_dtype(torch.float64)

assert torch.cuda.is_available()
DEVICE = torch.device("cuda")

D = 64
L = 3
STEPS = 100
LR = 2e-2
SEED = 0

# 谱动作：离散化程度（与层数 L 共同决定动作空间大小 3^L）
ACTIONS = (1.0, 0.5, 0.0)
ALL_SCHEDULES = list(cartesian_product(ACTIONS, repeat=L))

# 通用 DP 超参（不随训练阶段、调度类型变化）
ROLLOUT_H = 15
COMMIT_T = 10
# 可选：粗筛省算力（精筛目标仍是 rollout 末端 loss）
USE_COARSE_FINE = True
ROLLOUT_H_COARSE = 5
ROLLOUT_TOP_K = 5

print(f"device={DEVICE}, d={D}, |A|={len(ALL_SCHEDULES)}, H={ROLLOUT_H}, T={COMMIT_T}")

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_target(d, seed=0):
    gen = torch.Generator().manual_seed(seed)
    return torch.randn(d, d, generator=gen) / math.sqrt(d)


def rand_matrix(d, seed):
    state = torch.cuda.get_rng_state()
    torch.cuda.manual_seed(seed)
    W = torch.randn(d, d, device="cuda") / math.sqrt(d)
    torch.cuda.set_rng_state(state)
    return W


def product(Ws):
    P = torch.eye(Ws[0].shape[0], device=Ws[0].device, dtype=Ws[0].dtype)
    for W in Ws:
        P = W @ P
    return P


def prefix_products(Ws):
    d = Ws[0].shape[0]
    prefixes = [torch.eye(d, device=Ws[0].device, dtype=Ws[0].dtype)]
    P = prefixes[0]
    for W in Ws:
        P = W @ P
        prefixes.append(P)
    return prefixes


def suffix_products(Ws):
    L, d = len(Ws), Ws[0].shape[0]
    suffixes = [None] * L
    S = torch.eye(d, device=Ws[0].device, dtype=Ws[0].dtype)
    for i in reversed(range(L)):
        suffixes[i] = S.clone()
        S = S @ Ws[i]
    return suffixes


def loss(Ws, A):
    E = product(Ws) - A
    return 0.5 * (E * E).sum()


def grads_chain(Ws, A):
    prefixes = prefix_products(Ws)
    suffixes = suffix_products(Ws)
    E = prefixes[-1] - A
    return [suffixes[i].T @ E @ prefixes[i].T for i in range(len(Ws))]



def spectralized_update(U, p=1.0, eps=1e-12):
    nrm = torch.linalg.norm(U, ord="fro")
    if nrm < eps:
        return U.clone()
    P, s, Qt = torch.linalg.svd(U, full_matrices=False)
    U_new = P @ torch.diag((s + eps) ** p) @ Qt
    return U_new * (nrm / (torch.linalg.norm(U_new, ord="fro") + eps))


def greedy_step(Ws, A, lr=1e-3, p=1.0):
    return [W + spectralized_update(-lr * G, p=p) for W, G in zip(Ws, grads_chain(Ws, A))]


def spectral_step(Ws, A, lr, schedule):
    return [W + spectralized_update(-lr * G, p=p) for W, G, p in zip(Ws, grads_chain(Ws, A), schedule)]


def rollout_schedule(Ws, A, schedule, lr, horizon):
    Ws_roll = [W.clone() for W in Ws]
    for _ in range(horizon):
        Ws_roll = spectral_step(Ws_roll, A, lr, schedule)
    return loss(Ws_roll, A).item()


def rollout_horizon(step, total_steps, h_max):
    """只用剩余步数截断，不引入训练阶段逻辑。"""
    return max(1, min(total_steps - step, h_max))


def select_best_schedule(Ws, A, lr, schedules, horizon):
    best_loss = float("inf")
    best_schedule = schedules[0]
    for schedule in schedules:
        final_loss = rollout_schedule(Ws, A, schedule, lr, horizon)
        if final_loss < best_loss:
            best_loss = final_loss
            best_schedule = schedule
    return best_schedule


def select_best_schedule_coarse_fine(Ws, A, lr, schedules, h_coarse, h_fine, top_k):
    ranked = []
    for schedule in schedules:
        ranked.append((rollout_schedule(Ws, A, schedule, lr, h_coarse), schedule))
    ranked.sort(key=lambda x: x[0])
    finalists = [s for _, s in ranked[:top_k]]
    return select_best_schedule(Ws, A, lr, finalists, h_fine)


def run_experiment(seed=0, d=64, L=3, steps=100, lr=2e-2,
                   schedules=None, rollout_h=15, commit_t=10,
                   use_coarse_fine=True, h_coarse=5, top_k=5,
                   print_every=50, verbose=True, dtype=torch.float64):
    if schedules is None:
        schedules = ALL_SCHEDULES

    set_seed(seed)
    A = make_target(d, seed=seed).to(device=DEVICE, dtype=dtype)
    Ws0 = [rand_matrix(d, seed + i).to(dtype=dtype) for i in range(L)]

    methods = {
        "sgd": [W.clone() for W in Ws0],
        "dp": [W.clone() for W in Ws0],
    }
    history = {
        "sgd": {"loss": []},
        "dp": {"loss": [], "spectral_schedule": []},
    }

    dp_schedule = None

    for t in range(steps):
        for name, Ws in methods.items():
            history[name]["loss"].append(loss(Ws, A).item())

        if verbose and print_every is not None and t % print_every == 0:
            msg = f"step {t:4d}"
            for name in methods:
                msg += f" | {name}: loss={history[name]['loss'][-1]:.4f}"
            print(msg)

        methods["sgd"] = greedy_step(methods["sgd"], A, lr=lr, p=1.0)

        if dp_schedule is None or t % commit_t == 0:
            h = rollout_horizon(t, steps, rollout_h)
            h_c = min(h, h_coarse)
            if use_coarse_fine and len(schedules) > top_k:
                dp_schedule = select_best_schedule_coarse_fine(
                    methods["dp"], A, lr, schedules, h_c, h, top_k,
                )
            else:
                dp_schedule = select_best_schedule(methods["dp"], A, lr, schedules, h)

        methods["dp"] = spectral_step(methods["dp"], A, lr, dp_schedule)
        history["dp"]["spectral_schedule"].append(tuple(dp_schedule))

    for name, Ws in methods.items():
        history[name]["loss"].append(loss(Ws, A).item())

    if verbose:
        print("\n=== Final training metrics ===")
        for name, Ws in methods.items():
            print(f"\n{name}")
            print("final loss          =", history[name]["loss"][-1])
            if name == "dp":
                print("spectral schedule counts:")
                for k, v in Counter(history["dp"]["spectral_schedule"]).items():
                    print("  ", k, ":", v)
    return history, methods, A

In [ ]:
def plot_training_curves(history, title="Training curves"):
  colors = {"sgd": "#243447", "dp": "#2F80ED"}

  fig, ax = plt.subplots(figsize=(7.2, 4.2))
  for name in history:
    ax.plot(history[name]["loss"], label=name, color=colors.get(name, "gray"), linewidth=2.0)
  ax.set_xlabel("step")
  ax.set_ylabel("loss")
  ax.set_title("Terminal loss")
  ax.legend(frameon=False)
  ax.grid(True, alpha=0.25)
  fig.suptitle(title)
  plt.tight_layout()
  plt.show()


In [ ]:
history, methods, A = run_experiment(
  seed=SEED, d=D, L=L, steps=STEPS, lr=LR,
  schedules=ALL_SCHEDULES,
  rollout_h=ROLLOUT_H,
  commit_t=COMMIT_T,
  use_coarse_fine=USE_COARSE_FINE,
  h_coarse=ROLLOUT_H_COARSE,
  top_k=ROLLOUT_TOP_K,
  print_every=50,
)

In [ ]:
plot_training_curves(history, title=f"SGD vs DP (|A|={len(ALL_SCHEDULES)}, H={ROLLOUT_H}, T={COMMIT_T})")


## Jacobian value estimate

Let the current error be
$$
e=\operatorname{vec}(W_3W_2W_1-A).
$$
For a candidate schedule $a=(p_1,p_2,p_3)$, first apply one scheduled update and denote the resulting parameters by $W^+(a)$ and the one-step loss by
$$
L^+(a)=\tfrac12\|e^+(a)\|_2^2.
$$
Now locally linearize the error around $W^+(a)$ with respect to the parameters that will continue to train. Let $J^+(a)$ be this Jacobian and $K^+(a)=J^+(a)J^+(a)^\top$. The effective contraction rate in the current error direction is
$$
\lambda_{\mathrm{eff}}^+(a)
=
\frac{e^+(a)^\top K^+(a)e^+(a)}{e^+(a)^\top e^+(a)}
=
\frac{\|J^+(a)^\top e^+(a)\|_2^2}{\|e^+(a)\|_2^2}.
$$
The last form is what the code uses: $J^\top e$ is exactly the gradient of the squared-error loss with respect to the trainable weights, so we can compute $\lambda_{\mathrm{eff}}$ from ordinary gradient norms without explicitly materializing the full Jacobian Gram matrix.

If the local geometry changes slowly for the next $H-1$ steps, the predicted loss after horizon $H$ is
$$
\widehat L_H(a)
=
L^+(a)\exp\left[-2\eta(H-1)\lambda_{\mathrm{eff}}^+(a)\right].
$$
This yields a concrete trade-off test against a baseline action $b$:
$$
\log\frac{L^+(a)}{L^+(b)}
<
2\eta(H-1)\left[\lambda_{\mathrm{eff}}^+(a)-\lambda_{\mathrm{eff}}^+(b)\right].
$$
The left side is the immediate loss penalty. The right side is the predicted future convergence-rate gain. A spectrum-shaping action is attractive when its future rate gain is larger than its short-term cost.


## Three-way comparison

The next cell runs the three methods on the same problem. For `dp_jacobian`, the printed decision rows show why a schedule was selected: `penalty_vs_sgd` is the immediate loss cost relative to the SGD action, `rate_gain_vs_sgd` is the predicted future rate advantage, and `pred_log_gain` is their difference.


In [ ]:
JAC_RATE_WEIGHT = 1.0
JAC_COMPARE_STEPS = 50


def jacobian_lambda_eff(Ws, A, trainable_from=0, eps=1e-12):
    # Return e^T J J^T e / e^T e without explicitly building J J^T.
    E = product(Ws) - A
    denom = (E * E).sum().item() + eps
    grads = grads_chain(Ws, A)[trainable_from:]
    numer = sum((G * G).sum().item() for G in grads)
    return numer / denom


def jacobian_schedule_score(Ws, A, schedule, lr, horizon, rate_weight=JAC_RATE_WEIGHT):
    Ws_next = spectral_step(Ws, A, lr, schedule)
    immediate = loss(Ws_next, A).item()
    lam = jacobian_lambda_eff(Ws_next, A)
    future_h = max(0, horizon - 1)
    predicted = immediate * math.exp(-2.0 * rate_weight * lr * future_h * lam)
    return predicted, immediate, lam


def select_best_schedule_jacobian(Ws, A, lr, schedules, horizon, rate_weight=JAC_RATE_WEIGHT):
    rows = []
    for schedule in schedules:
        predicted, immediate, lam = jacobian_schedule_score(
            Ws, A, schedule, lr, horizon, rate_weight=rate_weight,
        )
        rows.append((predicted, immediate, lam, tuple(schedule)))
    rows.sort(key=lambda x: x[0])
    return rows[0][3], rows


def tradeoff_certificate(best_row, baseline_row, lr, horizon):
    _, best_immediate, best_lam, best_schedule = best_row
    _, base_immediate, base_lam, base_schedule = baseline_row
    future_h = max(0, horizon - 1)
    immediate_penalty = math.log((best_immediate + 1e-12) / (base_immediate + 1e-12))
    rate_gain = 2.0 * lr * future_h * (best_lam - base_lam)
    predicted_log_gain = rate_gain - immediate_penalty
    return dict(
        chosen=best_schedule,
        baseline=base_schedule,
        immediate_penalty=immediate_penalty,
        rate_gain=rate_gain,
        predicted_log_gain=predicted_log_gain,
    )


def run_experiment_with_jacobian_value(
    seed=0, d=64, L=3, steps=50, lr=2e-2,
    schedules=None, rollout_h=15, commit_t=10,
    rate_weight=JAC_RATE_WEIGHT, verbose=True,
):
    if schedules is None:
        schedules = ALL_SCHEDULES

    set_seed(seed)
    A = make_target(d, seed=seed).to(device=DEVICE, dtype=torch.float64)
    Ws0 = [rand_matrix(d, seed + i).to(dtype=torch.float64) for i in range(L)]

    methods = {
        "sgd": [W.clone() for W in Ws0],
        "dp_rollout": [W.clone() for W in Ws0],
        "dp_jacobian": [W.clone() for W in Ws0],
    }
    history = {name: {"loss": []} for name in methods}
    history["dp_rollout"]["spectral_schedule"] = []
    history["dp_jacobian"]["spectral_schedule"] = []
    history["dp_jacobian"]["decision_rows"] = []

    rollout_schedule_current = None
    jacobian_schedule_current = None

    for t in range(steps):
        for name, Ws in methods.items():
            history[name]["loss"].append(loss(Ws, A).item())

        methods["sgd"] = greedy_step(methods["sgd"], A, lr=lr, p=1.0)

        if rollout_schedule_current is None or t % commit_t == 0:
            h = rollout_horizon(t, steps, rollout_h)
            rollout_schedule_current = select_best_schedule(
                methods["dp_rollout"], A, lr, schedules, h,
            )
        methods["dp_rollout"] = spectral_step(methods["dp_rollout"], A, lr, rollout_schedule_current)
        history["dp_rollout"]["spectral_schedule"].append(tuple(rollout_schedule_current))

        if jacobian_schedule_current is None or t % commit_t == 0:
            h = rollout_horizon(t, steps, rollout_h)
            jacobian_schedule_current, rows = select_best_schedule_jacobian(
                methods["dp_jacobian"], A, lr, schedules, h, rate_weight=rate_weight,
            )
            best = rows[0]
            sgd_row = next(row for row in rows if row[3] == (1.0,) * L)
            immediate_best = min(rows, key=lambda row: row[1])
            cert = tradeoff_certificate(best, sgd_row, lr, h)
            history["dp_jacobian"]["decision_rows"].append(dict(
                step=t,
                horizon=h,
                chosen=best[3],
                predicted=best[0],
                immediate=best[1],
                lambda_eff=best[2],
                immediate_best=immediate_best[3],
                sgd_predicted=sgd_row[0],
                sgd_immediate=sgd_row[1],
                sgd_lambda=sgd_row[2],
                immediate_penalty_vs_sgd=cert["immediate_penalty"],
                rate_gain_vs_sgd=cert["rate_gain"],
                predicted_log_gain_vs_sgd=cert["predicted_log_gain"],
            ))

        methods["dp_jacobian"] = spectral_step(methods["dp_jacobian"], A, lr, jacobian_schedule_current)
        history["dp_jacobian"]["spectral_schedule"].append(tuple(jacobian_schedule_current))

    for name, Ws in methods.items():
        history[name]["loss"].append(loss(Ws, A).item())

    if verbose:
        print("\n=== Jacobian-guided value comparison ===")
        for name in ["sgd", "dp_rollout", "dp_jacobian"]:
            print(f"{name:12s}: final loss={history[name]['loss'][-1]:.6f}")
        print("\nJacobian decisions:")
        for row in history["dp_jacobian"]["decision_rows"]:
            print(
                f"  step {row['step']:3d}, H={row['horizon']:2d}, chosen={row['chosen']}, "
                f"immediate={row['immediate']:.4f}, lambda={row['lambda_eff']:.4f}, "
                f"penalty_vs_sgd={row['immediate_penalty_vs_sgd']:+.4f}, "
                f"rate_gain_vs_sgd={row['rate_gain_vs_sgd']:+.4f}, "
                f"pred_log_gain={row['predicted_log_gain_vs_sgd']:+.4f}"
            )
        print("\nSchedule counts:")
        for k, v in Counter(history["dp_jacobian"]["spectral_schedule"]).items():
            print("  ", k, ":", v)

    return history, methods, A


jac_history, jac_methods, jac_A = run_experiment_with_jacobian_value(
    seed=SEED, d=D, L=L, steps=JAC_COMPARE_STEPS, lr=LR,
    schedules=ALL_SCHEDULES, rollout_h=ROLLOUT_H, commit_t=COMMIT_T,
    rate_weight=JAC_RATE_WEIGHT, verbose=True,
)


In [ ]:
def plot_jacobian_value_comparison(history, title="SGD vs Rollout-MPC vs Jacobian value"):
    colors = {"sgd": "#243447", "dp_rollout": "#2F80ED", "dp_jacobian": "#D946EF"}
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    for name in ["sgd", "dp_rollout", "dp_jacobian"]:
        ax.plot(history[name]["loss"], label=name, color=colors[name], linewidth=2.0)
    ax.set_xlabel("step")
    ax.set_ylabel("loss")
    ax.set_title("loss")
    ax.legend(frameon=False)
    ax.grid(True, alpha=0.25)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_jacobian_value_comparison(
    jac_history,
    title=f"50-step Jacobian-guided search (|A|={len(ALL_SCHEDULES)}, H={ROLLOUT_H}, T={COMMIT_T})",
)


## Fast evaluation of proposed fixed schedules

Now consider the practical question: if we are given several candidate spectral modes, can we rank them without fully training every candidate? In this section, each candidate is a **fixed schedule**
$$
a=(p_1,p_2,p_3),
$$
which means the same spectral mode is used at every executed step. This is deliberately simpler than a fully time-varying plan: the goal here is to isolate whether a short prefix plus the Jacobian value estimate can predict which mode is more promising over a longer horizon.

The exact evaluation of a fixed schedule is simple but expensive: start from the same initial weights, execute the schedule for all $T$ steps, and compare the terminal loss $L_T$. The fast evaluator replaces this full run by a prefix run plus a local value estimate:

1. Execute the candidate schedule for only $B<T$ steps and reach $W_B(a)$.
2. Record the prefix loss
$$
L_B(a)=\frac12\|F(W_B(a))-A\|_F^2,
$$
where $F(W)=W_3W_2W_1$ in this notebook.
3. At $W_B(a)$, compute the local effective contraction rate
$$
\lambda_{\mathrm{eff}}(W_B(a))
=
\frac{e_B^\top J_BJ_B^\top e_B}{e_B^\top e_B},
\qquad
e_B=\operatorname{vec}(F(W_B(a))-A).
$$
In the linear-network case this quantity can be computed from the layer gradients, so we do not need to explicitly materialize $J_BJ_B^\top$.
4. Estimate the remaining $T-B$ steps by assuming the local contraction rate is approximately persistent, with a discount $\alpha\in(0,1]$:
$$
\widehat L_T(a)
=
L_B(a)\exp\{-2\alpha\eta (T-B)\lambda_{\mathrm{eff}}(W_B(a))\}.
$$

The algorithm is:

```text
Input:
  candidate fixed schedules a_1, ..., a_m
  target horizon T
  prefix budget B < T
  learning rate eta
  geometry discount alpha

For each candidate schedule a_i:
  Reset to the shared initial weights W_0

  For t = 0, ..., B-1:
    Apply one spectral update using the same schedule a_i

  Measure the reached state W_B(a_i)
  Compute prefix loss L_B(a_i)
  Compute local effective contraction lambda_eff(W_B(a_i))

  Predict terminal loss:
    L_hat_T(a_i) = L_B(a_i) * exp(-2 * alpha * eta * (T-B) * lambda_eff(W_B(a_i)))

Rank all fixed schedules by L_hat_T(a_i), smaller is better.

Optional validation only:
  Fully execute each a_i for T steps and compare true L_T with L_hat_T.
```

Finally, rank candidate schedules by $\widehat L_T(a)$. The optional `true_final` column in the code below is only a sanity check: it fully runs each schedule so we can see whether the fast ranking agrees with the actual terminal loss. In a real search loop, we would use the fast estimate to shortlist promising schedules and reserve full rollout for a much smaller set.

In [ ]:
import time


FIXED_TARGET_STEPS = 20
FIXED_PREFIX_STEPS = 1
FIXED_ALPHA = 0.05

PROPOSED_SCHEDULES = [
    (1.0, 1.0, 1.0),
    (0.5, 0.5, 0.5),
    (0.0, 0.0, 0.0),
    (1.0, 0.5, 1.0),
    (0.5, 0.0, 0.5),
    (0.0, 0.5, 0.0),
    (1.0, 0.0, 1.0),
    (0.0, 1.0, 0.0),
]


def clone_weights(Ws):
    return [W.clone() for W in Ws]


def make_problem(seed=SEED, d=D, L=L, dtype=torch.float64):
    set_seed(seed)
    A = make_target(d, seed=seed).to(device=DEVICE, dtype=dtype)
    Ws0 = [rand_matrix(d, seed + i).to(dtype=dtype) for i in range(L)]
    return Ws0, A


def run_fixed_schedule(Ws0, A, schedule, steps, lr=LR):
    Ws = clone_weights(Ws0)
    losses = [loss(Ws, A).item()]
    for _ in range(steps):
        Ws = spectral_step(Ws, A, lr, schedule)
        losses.append(loss(Ws, A).item())
    return Ws, losses


def rankdata(values):
    order = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(values):
        j = i + 1
        while j < len(values) and values[order[j]] == values[order[i]]:
            j += 1
        avg = 0.5 * (i + j - 1)
        for k in range(i, j):
            ranks[order[k]] = avg
        i = j
    return ranks


def spearmanr(xs, ys):
    rx, ry = rankdata(xs), rankdata(ys)
    mx = sum(rx) / len(rx)
    my = sum(ry) / len(ry)
    vx = sum((x - mx) ** 2 for x in rx)
    vy = sum((y - my) ** 2 for y in ry)
    if vx == 0 or vy == 0:
        return float("nan")
    return sum((x - mx) * (y - my) for x, y in zip(rx, ry)) / math.sqrt(vx * vy)


def fast_fixed_schedule_evaluation(
    candidate_schedules=PROPOSED_SCHEDULES,
    target_steps=FIXED_TARGET_STEPS,
    prefix_steps=FIXED_PREFIX_STEPS,
    alpha=FIXED_ALPHA,
    seed=SEED,
    d=D,
    L=L,
    lr=LR,
    run_truth=True,
):
    Ws0, A = make_problem(seed=seed, d=d, L=L)
    prefix_steps = min(prefix_steps, target_steps)
    remaining = target_steps - prefix_steps

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    rows = []
    for schedule in candidate_schedules:
        Ws_prefix, prefix_losses = run_fixed_schedule(Ws0, A, schedule, prefix_steps, lr=lr)
        lam = jacobian_lambda_eff(Ws_prefix, A)
        predicted = prefix_losses[-1] * math.exp(-2.0 * alpha * lr * remaining * lam)
        one_step_loss = run_fixed_schedule(Ws0, A, schedule, 1, lr=lr)[1][-1]
        rows.append(dict(
            schedule=tuple(schedule),
            one_step_loss=one_step_loss,
            prefix_loss=prefix_losses[-1],
            lambda_eff=lam,
            predicted_final=predicted,
        ))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    predictor_time = time.perf_counter() - t0

    truth_time = None
    if run_truth:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        for row in rows:
            _, full_losses = run_fixed_schedule(Ws0, A, row["schedule"], target_steps, lr=lr)
            row["true_final"] = full_losses[-1]
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        truth_time = time.perf_counter() - t1

    rows_by_pred = sorted(rows, key=lambda r: r["predicted_final"])
    rows_by_prefix = sorted(rows, key=lambda r: r["prefix_loss"])
    rows_by_one_step = sorted(rows, key=lambda r: r["one_step_loss"])
    print(f"target_steps={target_steps}, prefix_steps={prefix_steps}, alpha={alpha}")
    print(f"fast predictor time={predictor_time:.3f}s")
    if truth_time is not None:
        print(f"full fixed-schedule check time={truth_time:.3f}s, speedup={truth_time / predictor_time:.2f}x")
        rho_prefix = spearmanr([r["prefix_loss"] for r in rows], [r["true_final"] for r in rows])
        rho_pred = spearmanr([r["predicted_final"] for r in rows], [r["true_final"] for r in rows])
        print(f"Spearman(prefix-only, true)={rho_prefix:+.3f}")
        print(f"Spearman(predicted, true)={rho_pred:+.3f}")
    print(f"one-step greedy best: {rows_by_one_step[0]['schedule']}")
    print(f"prefix-only best: {rows_by_prefix[0]['schedule']}")
    print(f"prefix-Jacobian best: {rows_by_pred[0]['schedule']}")

    header = "rank | schedule        | one-step | prefix | lambda_eff | predicted"
    if run_truth:
        header += " | true"
    print("\n" + header)
    for k, row in enumerate(rows_by_pred, 1):
        line = (
            f"{k:4d} | {str(row['schedule']):15s} | "
            f"{row['one_step_loss']:.4f} | {row['prefix_loss']:.4f} | "
            f"{row['lambda_eff']:.4f} | {row['predicted_final']:.4f}"
        )
        if run_truth:
            line += f" | {row['true_final']:.4f}"
        print(line)
    return rows, dict(predictor_time=predictor_time, truth_time=truth_time)


fixed_rows, fixed_timing = fast_fixed_schedule_evaluation()


In [ ]:
def plot_fixed_schedule_evaluation(rows):
    ordered = sorted(rows, key=lambda r: r["predicted_final"])
    labels = ["".join(str(int(2 * p)) for p in r["schedule"]) for r in ordered]
    x = list(range(len(ordered)))
    width = 0.36

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
    axes[0].bar([i - width / 2 for i in x], [r["predicted_final"] for r in ordered],
                width=width, label="predicted", color="#2F80ED")
    if "true_final" in ordered[0]:
        axes[0].bar([i + width / 2 for i in x], [r["true_final"] for r in ordered],
                    width=width, label="true", color="#F2994A")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].set_xlabel("schedule code: 2=1.0, 1=0.5, 0=0.0")
    axes[0].set_ylabel("final loss")
    axes[0].set_title("Predicted vs true terminal loss")
    axes[0].legend(frameon=False)
    axes[0].grid(True, axis="y", alpha=0.25)

    if "true_final" in ordered[0]:
        axes[1].scatter([r["predicted_final"] for r in ordered],
                        [r["true_final"] for r in ordered],
                        s=60, color="#27AE60")
        for label, row in zip(labels, ordered):
            axes[1].annotate(label, (row["predicted_final"], row["true_final"]),
                             textcoords="offset points", xytext=(4, 4), fontsize=9)
        axes[1].set_xlabel("predicted final loss")
        axes[1].set_ylabel("true final loss")
        axes[1].set_title("Ranking sanity check")
        axes[1].grid(True, alpha=0.25)
    else:
        axes[1].axis("off")

    plt.tight_layout()
    plt.show()


plot_fixed_schedule_evaluation(fixed_rows)


## Reading the result

The Jacobian-guided policy usually starts with more aggressive spectrum shaping because the predicted future rate gain is large enough to compensate for the one-step loss penalty. As training progresses, the rate advantage shrinks, and the chosen schedules move closer to the SGD action.

The fixed-schedule screening experiment below is configured to show why the Jacobian term is useful. With prefix budget $B=1$, prefix-only screening is exactly the same kind of signal as one-step greedy selection: it asks which schedule gives the smallest loss after one update. In this run, that greedy criterion prefers the SGD-like schedule $(1,1,1)$.

The true best fixed schedule over the longer horizon is different. The prefix-Jacobian estimate uses the same one executed step, but also evaluates the local contraction rate at the reached state. That future-contraction term changes the ranking and selects $(0.5,0.5,0.5)$, which matches the full $T$-step check.

The discount $\\alpha$ is important. Without it, a single local Jacobian can overestimate schedules whose favorable geometry is short-lived. With a small discounted remaining-horizon estimate, the prediction is still cheap, but it can look beyond the immediate loss drop.